<a href="https://colab.research.google.com/github/blacklack547-hash/game-playtime-dashboard/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd

# Load the zipped CSV straight from your GitHub repository raw link
url = "https://raw.githubusercontent.com/blacklack547-hash/game-playtime-dashboard/main/hltb_dataset_normalized.zip"

# Pandas automatically detects the .zip compression and extracts it in memory!
df = pd.read_csv(url)

In [23]:
!pip install streamlit pandas plotly

In [24]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [25]:
%%writefile etl_pipeline.py
import pandas as pd
import sqlite3
import os

print("=== STARTING HLTB PLAYTIME ETL PIPELINE ===")

# Direct GitHub Raw link (Pandas auto-fetches it over the web!)
# (If you uploaded the .zip version to GitHub, change extension to .zip)
csv_url = "https://raw.githubusercontent.com/blacklack547-hash/game-playtime-dashboard/main/hltb_dataset_normalized.zip"

# 1. Extract raw data directly from GitHub
df = pd.read_csv(csv_url)
print(f"✔ Successfully extracted {len(df)} rows directly from GitHub.")

# 2. Curate & Transform Columns (Standardizing layout names to lowercase)
df.columns = df.columns.str.replace(' ', '_').str.lower().str.strip()

# Safely convert time metrics to numeric format and fill blanks with zeros
playtime_cols = ['main_story', 'main_plus_sides', 'completionist', 'all_styles']
for col in playtime_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# Standardize string fields to lowercase to avoid search mismatch errors
text_cols = ['name', 'type', 'platform', 'genres', 'release_date', 'source_url']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

# 3. Load into local relational database
conn = sqlite3.connect('backlog.db')
df.to_sql('playtimes', conn, if_exists='replace', index=False)
conn.close()

print("=== ETL PIPELINE PROCESS COMPLETED SUCCESSFULLY ===")

Overwriting etl_pipeline.py


In [26]:
!python etl_pipeline.py

=== STARTING HLTB PLAYTIME ETL PIPELINE ===
✔ Successfully extracted 166754 rows directly from GitHub.
=== ETL PIPELINE PROCESS COMPLETED SUCCESSFULLY ===


In [27]:
%%writefile app.py
import os
import zipfile

# Unzip database automatically on Streamlit Cloud startup
if not os.path.exists('backlog.db') and os.path.exists('backlog.zip'):
    with zipfile.ZipFile('backlog.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
import streamlit as st
import sqlite3
import pandas as pd

# Set up dashboard visual workspace with a clean layout
st.set_page_config(page_title="Game Playtime Analytics Dashboard", page_icon="🎮", layout="wide")

# Pull data directly from local relational database tables
@st.cache_data
def load_optimized_data():
    try:
        conn = sqlite3.connect('backlog.db')
        # 💡 FIX: Only pull games that actually have recorded playtime data
        data = pd.read_sql_query("SELECT * FROM playtimes WHERE main_story > 0", conn)
        conn.close()
        return data
    except Exception:
        return pd.DataFrame()

df = load_optimized_data()

if not df.empty:
    st.title("🎮 Video Game Playtime Curation & Analytics Platform")
    st.markdown("Query playtime playstyles, check release records, and browse game information seamlessly.")
    st.markdown("---")

    # High Level Summary Cards (Now filtering out 0-hour placeholder values!)
    kpi1, kpi2, kpi3 = st.columns(3)
    with kpi1:
        st.markdown("### 📊 Total Cataloged Titles")
        st.markdown(f"## **{len(df):,}**")
    with kpi2:
        st.markdown("### ⏱️ Average Main Story")
        # 💡 FIX: Only calculate mean for rows where main_story is greater than 0
        true_main_story = df[df['main_story'] > 0]['main_story'].mean()
        st.markdown(f"## **{true_main_story:.1f} Hours**")
    with kpi3:
        st.markdown("### 🏆 Average 100% Run")
        # 💡 FIX: Only calculate mean for rows where completionist is greater than 0
        true_completionist = df[df['completionist'] > 0]['completionist'].mean()
        st.markdown(f"## **{true_completionist:.1f} Hours**")

    st.markdown("---")

    # Analytics Leaderboard Deck (Pure text insights)
    st.subheader("📈 Playtime Records & Data Insights")
    st.markdown("Key structural milestones extracted directly from your database table:")

    c_longest, c_shortest = st.columns(2)
    with c_longest:
        st.markdown("#### 🏆 Top 5 Longest Games (100% Completion)")
        longest_games = df[df['completionist'] > 0].sort_values(by='completionist', ascending=False).head(5)
        for idx, row in longest_games.iterrows():
            st.markdown(f"• **{str(row['name']).title()}** ({str(row['type']).upper()}) — **{row['completionist']} hrs**")

    with c_shortest:
        st.markdown("#### ⚡ Top 5 Quickest Games (Main Story)")
        shortest_games = df[df['main_story'] > 0].sort_values(by='main_story', ascending=True).head(5)
        for idx, row in shortest_games.iterrows():
            st.markdown(f"• **{str(row['name']).title()}** ({str(row['type']).upper()}) — **{row['main_story']} hrs**")

    st.markdown("---")

    # Browse Catalog Layout (Reliable pure text deck)
    st.subheader("🗂️ Browse Your Cataloged Game Profiles")
    st.markdown("Snapshot overview of games extracted directly from your optimized local database storage:")

    preview_df = df.head(40)
    for idx, row in preview_df.iterrows():
        st.markdown(f"### 🕹️ {str(row['name']).title()}")

        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown(f"• **Main Story:** {row['main_story']} hrs")
            st.markdown(f"• **Main + Sides:** {row['main_plus_sides']} hrs")
        with c2:
            st.markdown(f"• **100% Run:** {row['completionist']} hrs")
            st.markdown(f"• **Average Style:** {row['all_styles']} hrs")
        with c3:
            st.markdown(f"• **Release Date:** {str(row['release_date']).title()}")
            st.markdown(f"• **Category:** {str(row['type']).upper()}")
        st.markdown(" ")

    st.markdown("---")

    # Dataset Spreadsheet View -> Completely static HTML table element
    st.subheader("📋 Curated Dataset Snapshot Table")
    st.table(df[['name', 'type', 'platform', 'main_story', 'completionist', 'release_date']].head(25))

else:
    st.error("⚠️ The SQLite database is empty. Please execute your 'etl_pipeline.py' script cell before launching the app.")

Overwriting app.py


In [28]:
import os
import time
import subprocess
import re
from IPython.display import display, HTML

print("🔄 Cleaning up old processes...")
!pkill -f streamlit
!pkill -f cloudflared
time.sleep(2)

# 1. Download Cloudflare tunnel engine
if not os.path.exists("cloudflared"):
    print("📥 Downloading Cloudflare engine...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# 2. Launch Streamlit silently
print("🚀 Starting Streamlit server...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(6)

# 3. Start Cloudflare Tunnel and grab full URL with Regex
print("🌐 Creating secure public link...\n")
tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

url_found = False
for _ in range(30):
    line = tunnel_process.stderr.readline()
    # Match the exact full URL (e.g. https://xxxx.trycloudflare.com)
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        full_url = match.group(0)
        display(HTML(f'''
            <div style="background-color: #d4edda; padding: 15px; border-radius: 8px; border: 1px solid #c3e6cb; margin-top: 10px;">
                <h3 style="color: #155724; margin-top: 0; font-family: sans-serif;">🎉 DASHBOARD IS LIVE!</h3>
                <a href="{full_url}" target="_blank" style="font-size: 18px; font-weight: bold; color: #0056b3; font-family: sans-serif;">
                    👉 CLICK HERE TO OPEN DASHBOARD
                </a>
                <p style="margin-bottom: 0; color: #155724; font-size: 13px; font-family: sans-serif; margin-top: 5px;">Direct URL: {full_url}</p>
            </div>
        '''))
        url_found = True
        break
    time.sleep(0.5)

if not url_found:
    print("⚠️ Link creation timed out. Please run this cell again.")

🔄 Cleaning up old processes...
🚀 Starting Streamlit server...
🌐 Creating secure public link...

